# Exercise 9: Does conversion help the task?

*Scripts and Encoding · version 0.1.0*

> Choose a task with data in a Cyrillic-script Turkic language (for example POS tagging of Kazakh). Train and evaluate the same model on native-script and CTS versions of the same split. Report task accuracy and tokens per word for both. Is the conversion worth it?

The notebook uses UPOS tagging on the Kyrgyz KTMU treebank (Universal Dependencies), which has a separate training and test file. (Kazakh KTB has only a tiny training split; see Neural Morphology.) The tagger is a simple logistic regression over word, prefix, suffix and neighbour features, so that the comparison isolates the input representation.

**Setup.** `pip install "turkicnlp==0.5.0" requests scikit-learn tiktoken`. Cells that fetch data from the web need internet access.

In [ ]:
import requests
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_Kyrgyz-KTMU/master/ky_ktmu-ud-{}.conllu"

def read_split(split):
    sents, cur = [], []
    for line in requests.get(URL.format(split), timeout=60).text.splitlines():
        if not line.strip():
            if cur: sents.append(cur); cur = []
        elif not line.startswith("#"):
            f = line.split("\t")
            if f[0].isdigit():                       # skip multiword ranges and empty nodes
                cur.append((f[1], f[3]))
    if cur: sents.append(cur)
    return sents

train, test = read_split("train"), read_split("test")
print(f"train {len(train)} sentences / {sum(map(len, train))} words; test {len(test)} / {sum(map(len, test))}")

In [ ]:
to_cts = Transliterator("kir", Script.CYRILLIC, Script.COMMON_TURKIC)

def features(words, i):
    w = words[i].lower()
    f = {"w=" + w: 1, "prev=" + (words[i - 1].lower() if i else "<s>"): 1,
         "next=" + (words[i + 1].lower() if i + 1 < len(words) else "</s>"): 1}
    for k in (1, 2, 3, 4):
        f[f"suf{k}=" + w[-k:]] = 1
        f[f"pre{k}=" + w[:k]] = 1
    return f

def dataset(sents, convert):
    X, y = [], []
    for s in sents:
        words = [to_cts.transliterate(w) if convert else w for w, _ in s]
        X += [features(words, i) for i in range(len(words))]
        y += [tag for _, tag in s]
    return X, y

results = {}
for name, convert in [("native Cyrillic", False), ("CTS", True)]:
    Xtr, ytr = dataset(train, convert)
    Xte, yte = dataset(test, convert)
    vec = DictVectorizer()
    clf = LogisticRegression(max_iter=2000).fit(vec.fit_transform(Xtr), ytr)
    acc = (clf.predict(vec.transform(Xte)) == yte).mean()
    results[name] = acc
    print(f"{name:<16} UPOS accuracy {acc:.2%}  features {len(vec.vocabulary_):,}")

In [ ]:
# Tokens per word for an LLM tokenizer on the same test words (downloads the vocabulary once)
try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")
    words = [w for s in test for w, _ in s]
    for name, conv in [("native Cyrillic", lambda w: w), ("CTS", to_cts.transliterate)]:
        print(f"{name:<16} {sum(len(enc.encode(conv(w))) for w in words) / len(words):.2f} tokens/word")
except Exception as exc:
    print("Tokenizer not available here:", type(exc).__name__)

## Worked solution (run on 18 October 2026)

UD Kyrgyz-KTMU: 1,308 training sentences (12,357 words) and 1,222 test sentences (11,771 words).

| Representation | UPOS accuracy |
|---|---|
| Native Cyrillic | 89.36% |
| CTS | 89.30% |

**Interpretation.** For a model trained and tested on one language, conversion makes no difference: CTS is an almost one-to-one recoding of Kyrgyz Cyrillic, so the tagger sees the same distinctions under different letters (CTS merges very few Kyrgyz letters, so the features barely change). Run the token cell to compare tokens per word: the chapter's measurements show that CTS lowers LLM tokeniser fertility for Cyrillic-script languages, which matters for cost and context length, but fertility is not accuracy.

**Is it worth it?** Not for this monolingual setting. Conversion becomes worth testing when the representation is shared *across* languages: a model trained on several Turkic languages in CTS, or transfer from a Latin-script language, as in The Computational Phenotype's Exercise 7, where converting Kyrgyz to CTS raised a Turkish parser's LAS from 2.0 to 18.4. The decision should be made per task, with the same split for both representations and the task metric reported alongside token cost.